## Imports & Connection

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [2]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [3]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)

local_datasets = '/Users/rapido/local-datasets/customer/appography/demographic_personas/'
print(local_datasets)

data_dumper_path = '/Users/rapido/local-datasets/customer/appography/appography_data_dump/'

/Users/rapido/analytics/customer/Appography/demographic_personas
/Users/rapido/local-datasets/customer/appography/demographic_personas/


## Parameter & Datasets

In [4]:
start_date = '20240301' 
end_date = '20240331'

In [5]:
selected_rr_customer_query = f"""

        SELECT
            city_name city,
            customer_id as user_id
        FROM
            orders.order_logs_snapshot
        WHERE 
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND city_name in ('Bangalore', 'Chennai', 'Delhi', 'Hyderabad')
            AND channel_host = 'android'
        GROUP BY 1,2
"""

df_selected_rr_customer = pd.read_sql(selected_rr_customer_query, connection)
df_selected_rr_customer.to_csv(local_datasets + 'raw/selected_rr_customer_{}_{}.csv'.format(start_date, end_date), index=False)

In [6]:
df_selected_rr_customer = pd.read_csv(local_datasets + 'raw/selected_rr_customer_{}_{}.csv'\
                                      .format(start_date, end_date))

In [7]:
df_selected_rr_customer.head(3)

,city,user_id
0,Hyderabad,6555db62273540bbbc52d3fd
1,Hyderabad,5d1112f14253186bbafef8dc
2,Hyderabad,60500db65e55c3bf78ce5719


In [8]:
df_customers_data = pd.read_csv(local_datasets + 'processed/pan_customer_use_case_{}_{}.csv'\
                                .format(start_date, end_date))
df_customers_data['age'] = df_customers_data['age'].astype('int32')
df_customers_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13308157 entries, 0 to 13308156
Data columns (total 8 columns):
 #   Column                        Dtype 
---  ------                        ----- 
 0   user_id                       object
 1   age                           int32 
 2   gender                        object
 3   first_half_drop_primary       object
 4   first_half_drop_secondary     object
 5   second_half_pickup_primary    object
 6   second_half_pickup_secondary  object
 7   app_list_set                  object
dtypes: int32(1), object(7)
memory usage: 761.5+ MB


### Data dictionary


<b>user_id</b> - Customer ID<br>
<b>age & gender</b> - Custoemr age & gender<br>
<b>first_half_drop_primary</b> - Primary or most repeated drop location use cases of customers (Morning to noon)<br>
<b>first_half_drop_secondary</b> - Primary or most repeated drop location use cases of customers (noon to night)<br>
<b>second_half_pickup_primary</b> - Primary or most repeated pickup location use cases of customers (Morning to noon)<br>
<b>second_half_pickup_secondary</b> - Primary or most repeated pickup location use cases of customers (noon to night)<br>

In [9]:
df_customers_data.head(1)

,user_id,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list_set
0,5fed72c9bd3aea15ec38af3e,43,MALE,residential,Unknown,health_and_personal,NaN,"['paytm money', 'facebook', 'netflix', 'youtub..."


### Category

In [10]:
df_category = pd.read_csv('category_file.csv')
print(df_category.shape)
df_category.head(5)

(398, 4)


,app_name,categories_l1,categories_l2,categories_l3
0,hinge,dating,rest,dating
1,aisle,dating,rest,dating
2,bumble,dating,rest,dating
3,coffee meets bagel,dating,rest,dating
4,happn,dating,rest,dating


### Data work-around

In [11]:
df_data = pd.merge(df_selected_rr_customer,
                   df_customers_data,
                   how = 'inner',
                   on = ['user_id'])
df_data.head(3)

,city,user_id,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list_set
0,Hyderabad,6555db62273540bbbc52d3fd,0,MALE,Unknown,NaN,govt_amenity,Unknown,"['google sheets', 'jio cinema', 'youtube', 'fa..."
1,Hyderabad,5d1112f14253186bbafef8dc,29,MALE,residential,leisure,residential,household_needs,"['google maps', 'paytm money', 'myntra', 'snap..."
2,Hyderabad,60500db65e55c3bf78ce5719,24,MALE,residential,NaN,transit_station,NaN,"['bookmyshow', 'youtube', 'google maps', 'gmai..."


In [12]:
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r"'([^']*)'", text)]

df_data['app_list_set'] = df_data['app_list_set'].apply(lambda x: extract_and_trim(x))
df_data.head(1)

,city,user_id,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list_set
0,Hyderabad,6555db62273540bbbc52d3fd,0,MALE,Unknown,NaN,govt_amenity,Unknown,"[google sheets, jio cinema, youtube, facebook,..."


In [13]:
df_customers_data_explode = df_data.explode('app_list_set')
df_customers_data_explode['app_list_set'] = df_customers_data_explode['app_list_set'].str.lower()

df_customers_data_explode.head(4)

,city,user_id,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list_set
0,Hyderabad,6555db62273540bbbc52d3fd,0,MALE,Unknown,NaN,govt_amenity,Unknown,google sheets
0,Hyderabad,6555db62273540bbbc52d3fd,0,MALE,Unknown,NaN,govt_amenity,Unknown,jio cinema
0,Hyderabad,6555db62273540bbbc52d3fd,0,MALE,Unknown,NaN,govt_amenity,Unknown,youtube
0,Hyderabad,6555db62273540bbbc52d3fd,0,MALE,Unknown,NaN,govt_amenity,Unknown,facebook


In [14]:
df_data = pd.merge(df_customers_data_explode,
                   df_category[['app_name','categories_l3']],
                   how = 'left',
                   left_on = ['app_list_set'],
                   right_on = ['app_name'])
df_data.head(3)

,city,user_id,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list_set,app_name,categories_l3
0,Hyderabad,6555db62273540bbbc52d3fd,0,MALE,Unknown,NaN,govt_amenity,Unknown,google sheets,google sheets,rest
1,Hyderabad,6555db62273540bbbc52d3fd,0,MALE,Unknown,NaN,govt_amenity,Unknown,jio cinema,jio cinema,rest
2,Hyderabad,6555db62273540bbbc52d3fd,0,MALE,Unknown,NaN,govt_amenity,Unknown,youtube,youtube,rest


In [15]:
df_data_mapped = df_data\
                        .groupby(['city','user_id','age','gender',
                                  'first_half_drop_primary','first_half_drop_secondary',
                                  'second_half_pickup_primary','second_half_pickup_secondary'])\
                        .agg(
                             app_list = ('app_list_set', lambda x: list(set(x))),
                             categories_l3 = ('categories_l3', lambda x: list(set(x)))
                            )\
                        .reset_index()

df_data_mapped.head(4)

,city,user_id,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list,categories_l2
0,Bangalore,5737ca80ddbec220de03eba4,25,FEMALE,educational,residential,residential,educational,"[truecaller, pinterest, flipkart, ola, google ...","[delivery_food, rest, delivery_grocery, stream..."
1,Bangalore,573f28f39b0ffc283676f0f4,0,FEMALE,transit_station,leisure,residential,leisure,"[truecaller, linkedin, ola, swiggy, amazon pri...","[dating, delivery_food, rest, delivery_grocery..."
2,Bangalore,573f28f39b0ffc283676f169,35,MALE,govt_amenity,household_needs,food,leisure,"[bookmyshow, flipkart, youtube, ludo king, wha...","[delivery_food, rest, delivery_grocery, health..."
3,Bangalore,573f28f49b0ffc283676f2e1,31,MALE,office,residential,office,residential,"[airbnb, mobikwik, twitter, bookmyshow, linked...","[dating, delivery_food, rest, delivery_grocery..."


In [18]:
unique_categories = set(category for sublist in df_data_mapped['categories_l3'] for category in sublist)
print("Count of unique categories:", len(unique_categories))
unique_categories

Count of unique categories: 18


{'core_educational',
 'core_office',
 'core_social',
 'dating',
 'delivery_food',
 'delivery_grocery',
 'devotional',
 'driver_app',
 'educational',
 'fantasy_sports',
 'finance_investment',
 'gaming',
 'health',
 'office',
 'rest',
 'social',
 'streaming_music',
 'travel_ridehailing'}

In [19]:
for category in unique_categories:
    df_data_mapped[category] = df_data_mapped['categories_l3'].apply(lambda x: 1 if category in x else 0)
df_data_mapped.head(1)

,city,user_id,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list,categories_l3,dating,delivery_food,core_educational,rest,driver_app,devotional,delivery_grocery,health,social,streaming_music,office,gaming,finance_investment,travel_ridehailing,core_office,core_social,educational,fantasy_sports
0,Bangalore,5737ca80ddbec220de03eba4,25,FEMALE,educational,residential,residential,educational,"[truecaller, pinterest, flipkart, ola, google ...","[delivery_food, rest, delivery_grocery, stream...",0,1,0,1,0,0,1,0,0,1,1,0,0,1,0,1,0,0


In [21]:
print(df_data_mapped.shape)
df_data_mapped.columns

(402614, 28)


Index(['city', 'user_id', 'age', 'gender', 'first_half_drop_primary',
       'first_half_drop_secondary', 'second_half_pickup_primary',
       'second_half_pickup_secondary', 'app_list', 'categories_l3', 'dating',
       'delivery_food', 'core_educational', 'rest', 'driver_app', 'devotional',
       'delivery_grocery', 'health', 'social', 'streaming_music', 'office',
       'gaming', 'finance_investment', 'travel_ridehailing', 'core_office',
       'core_social', 'educational', 'fantasy_sports'],
      dtype='object')

In [25]:
df_data_mapped = df_data_mapped[['city','user_id','age','gender',
                                 'first_half_drop_primary','first_half_drop_secondary',
                                 'second_half_pickup_primary','second_half_pickup_secondary',
                                 'app_list','categories_l3',
                                 'core_educational','core_office','core_social',
                                 'dating', 'delivery_food', 'delivery_grocery', 'devotional', 
                                 'driver_app', 'educational', 'fantasy_sports', 
                                 'finance_investment', 'gaming', 'health', 
                                 'office', 'social', 'streaming_music', 'travel_ridehailing', 'rest']]
df_data_mapped.head(3)

,city,user_id,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list,categories_l3,core_educational,core_office,core_social,dating,delivery_food,delivery_grocery,devotional,driver_app,educational,fantasy_sports,fantasy_sports,finance_investment,gaming,health,office,social,streaming_music,travel_ridehailing,rest
0,Bangalore,5737ca80ddbec220de03eba4,25,FEMALE,educational,residential,residential,educational,"[truecaller, pinterest, flipkart, ola, google ...","[delivery_food, rest, delivery_grocery, stream...",0,0,1,0,1,1,0,0,0,0,0,0,0,0,1,0,1,1,1
1,Bangalore,573f28f39b0ffc283676f0f4,0,FEMALE,transit_station,leisure,residential,leisure,"[truecaller, linkedin, ola, swiggy, amazon pri...","[dating, delivery_food, rest, delivery_grocery...",0,1,1,1,1,1,0,0,0,0,0,1,0,0,1,1,1,1,1
2,Bangalore,573f28f39b0ffc283676f169,35,MALE,govt_amenity,household_needs,food,leisure,"[bookmyshow, flipkart, youtube, ludo king, wha...","[delivery_food, rest, delivery_grocery, health...",0,0,1,0,1,1,0,0,1,0,0,0,1,1,0,0,0,1,1


In [26]:
df_data_mapped.to_csv(local_datasets + 'processed/df_data_mapped{}_{}.csv'\
                      .format(start_date, end_date), index=False)

## Analysis 

In [27]:
df_data_mapped.groupby(['city']).user_id.nunique()

city
Bangalore    124742
Chennai       66316
Delhi         66893
Hyderabad    144663
Name: user_id, dtype: int64

student_category


- Student-HC
    - core_educational [yes] & core_office [no]
    - educational [yes] & core_office[no]
    - primary use-case [educational]
    - secondary use-case [educational] & core_office [no]
- Student-LC
    - age [<= 25] & age [not null] & core_office [no]
        - dating [yes] or gaming [yes]
        - office [no] and finance_investment
    



In [76]:
def student_category(row):
    if row['core_educational'] == 1 and row['core_office'] == 0 :
        return 'Student-HC'
    elif row['educational'] == 1 and row['core_office'] == 0 :
        return 'Student-HC'
    
    elif (row['first_half_drop_primary'] == 'educational') :
        return 'Student-HC'
    elif (row['first_half_drop_secondary'] == 'educational') and (row['core_office'] == 0) :
        return 'Student-HC'
    elif (row['second_half_pickup_primary'] == 'educational') :
        return 'Student-HC'
    elif (row['second_half_pickup_secondary'] == 'educational') and (row['core_office'] == 0) :
        return 'Student-HC'
    
    
    elif row['age'] <= 25 and row['age'] != 0 and row['core_office'] == 0 :    
        if row['dating'] == 1 or row['gaming'] == 1 :
            return 'Student-LC'
        elif row['office'] == 0 and row['finance_investment'] == 0 :
            return 'Student-LC'
        else:
            'Unknown'
    
    else:
        return 'Unknown'
        
        
        
def parent_category(row):
    if row['core_office'] == 1:
        return 'Parent-HC'
    elif row['core_office'] == 1 or row['office'] == 1 :
        return 'Parent-HC'
    
#     elif (row['first_half_drop_primary'] != 'educational') :
#         return 'Parent-HC'
#     elif (row['first_half_drop_secondary'] != 'educational') & (row['core_educational'] == 0) :
#         return 'Parent-HC'
#     elif (row['second_half_pickup_primary'] != 'educational') :
#         return 'Parent-HC'
#     elif (row['second_half_pickup_secondary'] != 'educational') & (row['core_educational'] == 0) :
#         return 'Parent-HC'
    
    elif row['age'] >= 35 :
        return 'Parent-LC'
    elif row['finance_investment'] == 1 :
        return 'Parent-LC'
    elif row['health'] == 1 :
        return 'Parent-LC'
    
    else:
        return 'Unknown'

In [77]:
df_data_mapped['student_tag'] = df_data_mapped.apply(student_category, axis=1)
df_data_mapped['parent_tag'] = df_data_mapped.apply(parent_category, axis=1)

In [78]:
df_data_mapped.groupby(['city','student_tag']).user_id.nunique()

city       student_tag
Bangalore  Student-HC     50004
           Student-LC      2458
           Unknown        70071
Chennai    Student-HC     30948
           Student-LC      1085
           Unknown        33361
Delhi      Student-HC     23958
           Student-LC      1981
           Unknown        38540
Hyderabad  Student-HC     59920
           Student-LC      3971
           Unknown        77555
Name: user_id, dtype: int64

In [79]:
df_data_mapped.groupby(['city','parent_tag']).user_id.nunique()

city       parent_tag
Bangalore  Parent-HC     80371
           Parent-LC     11801
           Unknown       32570
Chennai    Parent-HC     41034
           Parent-LC      6567
           Unknown       18715
Delhi      Parent-HC     40530
           Parent-LC      8411
           Unknown       17952
Hyderabad  Parent-HC     85018
           Parent-LC     15409
           Unknown       44236
Name: user_id, dtype: int64

In [80]:
df_data_mapped

,city,user_id,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list,categories_l3,core_educational,core_office,core_social,dating,delivery_food,delivery_grocery,devotional,driver_app,educational,fantasy_sports,fantasy_sports,finance_investment,gaming,health,office,social,streaming_music,travel_ridehailing,rest,student_tag,parent_tag,level_one_filter
0,Bangalore,5737ca80ddbec220de03eba4,25,FEMALE,educational,residential,residential,educational,"[truecaller, pinterest, flipkart, ola, google ...","[delivery_food, rest, delivery_grocery, stream...",0,0,1,0,1,1,0,0,0,0,0,0,0,0,1,0,1,1,1,Student-HC,Parent-HC,Parent-HC
1,Bangalore,573f28f39b0ffc283676f0f4,0,FEMALE,transit_station,leisure,residential,leisure,"[truecaller, linkedin, ola, swiggy, amazon pri...","[dating, delivery_food, rest, delivery_grocery...",0,1,1,1,1,1,0,0,0,0,0,1,0,0,1,1,1,1,1,Unknown,Parent-HC,Parent-HC
2,Bangalore,573f28f39b0ffc283676f169,35,MALE,govt_amenity,household_needs,food,leisure,"[bookmyshow, flipkart, youtube, ludo king, wha...","[delivery_food, rest, delivery_grocery, health...",0,0,1,0,1,1,0,0,1,0,0,0,1,1,0,0,0,1,1,Student-HC,Parent-LC,Student-HC
3,Bangalore,573f28f49b0ffc283676f2e1,31,MALE,office,residential,office,residential,"[airbnb, mobikwik, twitter, bookmyshow, linked...","[dating, delivery_food, rest, delivery_grocery...",0,1,1,1,1,1,0,0,1,0,0,1,0,0,1,1,1,1,1,Unknown,Parent-HC,Parent-HC
4,Bangalore,573f28f49b0ffc283676f300,28,MALE,household_needs,residential,household_needs,Unknown,"[booking.com, twitter, linkedin, flipkart, ola...","[delivery_food, rest, social, streaming_music,...",0,1,1,0,1,0,0,0,0,0,0,1,0,0,1,1,1,1,1,Unknown,Parent-HC,Parent-HC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
402609,Hyderabad,6608fa3175d2871fc9d63554,0,MALE,office,Unknown,transit_station,Unknown,"[ludo king, facebook, whatsapp, gmail, spotify...","[core_social, gaming, rest, streaming_music]",0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,Unknown,Unknown,Unknown
402610,Hyderabad,6608faaa23569d203c0a2092,0,MALE,residential,transit_station,residential,household_needs,"[youtube, google calendar, whatsapp, jio cinem...","[rest, core_social]",0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,Unknown,Unknown,Unknown
402611,Hyderabad,6608fb0fac5a66a503302c7d,0,MALE,residential,place_of_worship,leisure,Unknown,"[facebook, youtube, whatsapp, jio cinema, gmai...","[core_social, rest, office]",0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,Unknown,Parent-HC,Parent-HC
402612,Hyderabad,6608fd58db0e9cdf0e22f8ec,0,MALE,Unknown,office,health_and_personal,leisure,"[youtube, zomato, whatsapp, gmail, spotify, cl...","[delivery_food, driver_app, rest, streaming_mu...",0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,1,Unknown,Unknown,Unknown


In [87]:
def level_one_filter(row):
    if row['student_tag'] == row['parent_tag']:
        return row['student_tag']
    
    elif row['student_tag'] != 'Unknown' and row['parent_tag'] == 'Unknown':
        return row['student_tag']
    
    elif row['student_tag'] == 'Unknown' and row['parent_tag'] != 'Unknown':
        return row['parent_tag']
    
    elif row['student_tag'] == 'Student-HC' and row['parent_tag'] == 'Parent-HC':
        if row['office'] == 1 and row['core_educational'] == 0:
            return row['parent_tag']
        
    elif row['student_tag'] == 'Student-HC' and row['parent_tag'] == 'Parent-LC':
        return row['student_tag']
    
    elif row['student_tag'] != 'Student-HC' and (row['parent_tag'] == 'Parent-HC' or row['parent_tag'] == 'Parent-LC' ):
        return row['parent_tag']
    
    
    
    else:
        return 'check'
    
df_data_mapped['level_one_filter'] = df_data_mapped.apply(level_one_filter, axis=1)

In [89]:
df_data_mapped

,city,user_id,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list,categories_l3,core_educational,core_office,core_social,dating,delivery_food,delivery_grocery,devotional,driver_app,educational,fantasy_sports,fantasy_sports,finance_investment,gaming,health,office,social,streaming_music,travel_ridehailing,rest,student_tag,parent_tag,level_one_filter
0,Bangalore,5737ca80ddbec220de03eba4,25,FEMALE,educational,residential,residential,educational,"[truecaller, pinterest, flipkart, ola, google ...","[delivery_food, rest, delivery_grocery, stream...",0,0,1,0,1,1,0,0,0,0,0,0,0,0,1,0,1,1,1,Student-HC,Parent-HC,Parent-HC
1,Bangalore,573f28f39b0ffc283676f0f4,0,FEMALE,transit_station,leisure,residential,leisure,"[truecaller, linkedin, ola, swiggy, amazon pri...","[dating, delivery_food, rest, delivery_grocery...",0,1,1,1,1,1,0,0,0,0,0,1,0,0,1,1,1,1,1,Unknown,Parent-HC,Parent-HC
2,Bangalore,573f28f39b0ffc283676f169,35,MALE,govt_amenity,household_needs,food,leisure,"[bookmyshow, flipkart, youtube, ludo king, wha...","[delivery_food, rest, delivery_grocery, health...",0,0,1,0,1,1,0,0,1,0,0,0,1,1,0,0,0,1,1,Student-HC,Parent-LC,Student-HC
3,Bangalore,573f28f49b0ffc283676f2e1,31,MALE,office,residential,office,residential,"[airbnb, mobikwik, twitter, bookmyshow, linked...","[dating, delivery_food, rest, delivery_grocery...",0,1,1,1,1,1,0,0,1,0,0,1,0,0,1,1,1,1,1,Unknown,Parent-HC,Parent-HC
4,Bangalore,573f28f49b0ffc283676f300,28,MALE,household_needs,residential,household_needs,Unknown,"[booking.com, twitter, linkedin, flipkart, ola...","[delivery_food, rest, social, streaming_music,...",0,1,1,0,1,0,0,0,0,0,0,1,0,0,1,1,1,1,1,Unknown,Parent-HC,Parent-HC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
402609,Hyderabad,6608fa3175d2871fc9d63554,0,MALE,office,Unknown,transit_station,Unknown,"[ludo king, facebook, whatsapp, gmail, spotify...","[core_social, gaming, rest, streaming_music]",0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,Unknown,Unknown,Unknown
402610,Hyderabad,6608faaa23569d203c0a2092,0,MALE,residential,transit_station,residential,household_needs,"[youtube, google calendar, whatsapp, jio cinem...","[rest, core_social]",0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,Unknown,Unknown,Unknown
402611,Hyderabad,6608fb0fac5a66a503302c7d,0,MALE,residential,place_of_worship,leisure,Unknown,"[facebook, youtube, whatsapp, jio cinema, gmai...","[core_social, rest, office]",0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,Unknown,Parent-HC,Parent-HC
402612,Hyderabad,6608fd58db0e9cdf0e22f8ec,0,MALE,Unknown,office,health_and_personal,leisure,"[youtube, zomato, whatsapp, gmail, spotify, cl...","[delivery_food, driver_app, rest, streaming_mu...",0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,1,Unknown,Unknown,Unknown


In [95]:
df_analysis1 =  df_data_mapped\
                        .groupby(['city','level_one_filter'])\
                        .agg(customer = ('user_id', 'nunique'))\
                        .reset_index()

df_analysis1['total_customer'] = df_analysis1.groupby('city')['customer'].transform('sum')
df_analysis1['distr %'] = (df_analysis1['customer']*100.00/df_analysis1['total_customer']).round(2)
df_analysis1

,city,level_one_filter,customer,total_customer,distr %
0,Bangalore,Parent-HC,73646,118017,62.40
1,Bangalore,Parent-LC,5872,118017,4.98
2,Bangalore,Student-HC,19137,118017,16.22
3,Bangalore,Student-LC,2265,118017,1.92
4,Bangalore,Unknown,17097,118017,14.49
5,Chennai,Parent-HC,36326,61608,58.96
6,Chennai,Parent-LC,3204,61608,5.20
7,Chennai,Student-HC,12077,61608,19.60
8,Chennai,Student-LC,1020,61608,1.66
9,Chennai,Unknown,8981,61608,14.58


In [96]:
df_analysis1.pivot(index='level_one_filter', columns='city', values=['distr %'])

distr %                         
city             Bangalore Chennai  Delhi Hyderabad
level_one_filter                                   
Parent-HC            62.40   58.96  58.21     56.54
Parent-LC             4.98    5.20   8.34      6.10
Student-HC           16.22   19.60  13.27     17.47
Student-LC            1.92    1.66   2.71      2.78
Unknown              14.49   14.58  17.47     17.11

In [ ]:
df.pivot(index ='row_column' , columns ='column_name', values =['value'])
